## Data generation

In [1]:
import numpy as np
import os
import csv
import matplotlib.pyplot as plt

from Simu_Optical_Conjugation import *
from Zenital_tilt import *
from Simu_Rayleigh import *
from Simu_Berry import *
from Simu_Sky_Intensity_CIE import *
from Simu_Micro_Polarizers import *
from Simu_Sensor import *
from Simu_Data_Processing import *

Sensor

In [2]:
# pixel size in micrometers
pixel_size = 3.45
# focal length in millimeters
f = 8
# sensor high in pixels
sensor_rows = 2048
# sensor length in pixels
sensor_cols = 2448
# conjugation type
r_conj = 'r0'


OPTICS SIMULATION

In [3]:
# np.transpose(np.ones(3)[:,np.newaxis])
(np.ones(3)[np.newaxis,:]) # ligne
(np.ones(3)[:,np.newaxis]) # col
particules_azimuth_matrix_rad,particules_elevation_matrix_rad,X_coordinate_pixels,Y_coordinate_pixels = Simu_Optical_Conjugation(pixel_size,sensor_cols,sensor_rows,f,r_conj,0,0)
x_pixel_mesh = (np.ones((Y_coordinate_pixels.size))[:,np.newaxis]) * X_coordinate_pixels[np.newaxis,:]
y_pixel_mesh = np.ones(X_coordinate_pixels.size)[np.newaxis,:]*((Y_coordinate_pixels))[:,np.newaxis]


Save reference (untilted) geometry

In [4]:
particules_azimuth_matrix_ref = particules_azimuth_matrix_rad.copy()
particules_elevation_matrix_ref = particules_elevation_matrix_rad.copy()


Output directory

In [5]:
output_dir = os.path.join(os.getcwd(), "Simulation_Results")
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "simulation_metadata.csv")

if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "iteration",
            "time",
            "Psi_true_deg",
            "Psi_sn_deg",
            "zen_deg",
            "rotation_axis_azimuth_deg",
            "rotation_angle_deg",
            "data_file"
        ])


AHRS inputs

In [6]:
pose_csv_path = os.path.join(os.getcwd(), r"Simulation_Data\Simulation_Data.csv")

time_vec = []
True_heading_vec = []
rotation_angle_deg_vec = []
rotation_axis_azimuth_deg_vec = []

with open(pose_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        time_vec.append(float(row["time"])) # time
        True_heading_vec.append(float(row["yaw"])) # Psi_cn (degrees)
        rotation_angle_deg_vec.append(float(row["rotation_angle_deg"])) # Missalignment Disturbance
        rotation_axis_azimuth_deg_vec.append(float(row["rotation_axis_azimuth_deg"]))

# Tehran latitude and longitude, 10th of July 2025, 08:00 AM
Psi_sn =  86.2
zen = 55.2

assert len(True_heading_vec) == len(rotation_axis_azimuth_deg_vec) == len(rotation_angle_deg_vec), \
    "Parameter vectors must have the same length (coupled pose rows)"

# Loop

In [7]:
for iter in range(len(True_heading_vec)):
    
    ## ----- Tilt -----

    time = time_vec[iter]
    heading = True_heading_vec[iter]
    rotation_axis_azimuth_rad = (np.pi / 180) * rotation_axis_azimuth_deg_vec[iter]
    rotation_angle_rad = (np.pi / 180) * rotation_angle_deg_vec[iter]
    particules_azimuth_matrix_rad, particules_elevation_matrix_rad = Zenital_tilt(particules_azimuth_matrix_ref, particules_elevation_matrix_ref, rotation_axis_azimuth_rad, rotation_angle_rad)
    
    ## ----- Rayleigh -----
    
    sun_azimuth_deg = (heading+Psi_sn) % 360 # Psi_sc
    # sun elevation, in degree, from the ground (0) to the vertical z axis (or
    # zenith (90)
    sun_elevation_deg = 90-zen
    # maximum DoLP observable in sky, 0.75 for usual clear sky
    # Quality of Polarization Disturbance
    DoLP_Max = 0.75
    ## RAYLEIGHT'S MODEL SIMULATION
    sun_elevation_rad = sun_elevation_deg * np.pi / 180
    sun_azimuth_rad = sun_azimuth_deg * np.pi / 180
    # AoP_Matrix_Global_rad_Rayleigh,DoLP_Matrix_Rayleigh = Simu_Rayleigh(sun_elevation_rad,sun_azimuth_rad,particules_elevation_matrix_rad,particules_azimuth_matrix_rad,DoLP_Max)

    ## ----- Berry -----

    sun_azimuth_deg
    # sun elevation, in degree, from the ground (0) to the vertical z axis (or
    # zenith (90)
    sun_elevation_deg
    # maximum DoP observable in sky, 0.75 for usual clear sky
    DoLP_Max
    # angle between Babinet and Brewster points in degrees:
    delta_deg = 140
    sun_elevation_rad = sun_elevation_deg * np.pi / 180
    sun_azimuth_rad = sun_azimuth_deg * np.pi / 180
    delta_rad = delta_deg * np.pi / 180
    AoP_Matrix_Global_rad_Berry,DoLP_Matrix_Berry = Simu_Berry(sun_elevation_rad,sun_azimuth_rad,particules_elevation_matrix_rad,particules_azimuth_matrix_rad,delta_rad,DoLP_Max)

    ## ----- Sky radiance simulation -----

    # sun azimuth is expressed in x,y camera image plane, seen from above.
    # xyz is direct frame, z is up true vertical axis.
    # seen from outputed camera image, y axis goes up on the image, x axis goes
    # left on the image
    sun_azimuth_deg
    # sun elevation, in degree, from the ground (0) to the vertical z axis (or
    # zenith (90)
    sun_elevation_deg
    # CIE table sky model number, refere to the Article
    # "Analysis of vertical sky components under various CIE standard general skies" from D.H.W. Li, C. Li, S.W. Lou, E.K.W. Tsang and J.C. Lam wrote in 2015
    CIE_Sky_number = 10 # Sky Disturbance
    Skylight_Relative_Intensity = Simu_Sky_Intensity_CIE(sun_azimuth_rad,sun_elevation_rad,particules_azimuth_matrix_rad,particules_elevation_matrix_rad,CIE_Sky_number)

    ## ----- Micro polarizers simulation -----

    # The sensor called in this simulation  is based on sony IMX250MZR sensor used in FLIR BFS-U3-51S5P-C camera.
    # It is a sensor with a micro-polarizer array, each block of 2 by 2 pixels possess 4 type of linear micro-polarizer oriented at 0, 45, 90, and 135 degrees.
    # Here we compute micro-polarizer array effects
    # we already set size in OPITCS SIMULATION INPUTS:
    sensor_rows
    sensor_cols
    # each polarizer is not realy at its supposed orientation, here we put the mechanical tolerance in polarizer orientation in degrees
    tolerance_deg = 1
    # If T1 is the intensity transmitance for an incident ray totaly linearly
    # polarized along transmission axis and T2 is the intensity transmittance
    # for an incident ray totally linearly polarized at 90 degrees from
    # transmission axis, then the extinction rassio is (T1-T2)/(T1+T2).
    # Because we work with relative intensity their is no need to take into
    # account absorbance, so we suppose null absorbance, which means T1+T2=1.
    extinction_ratio = 0.99
    ## MICRO POLARIZER SIMULATION
    tolerance_rad = tolerance_deg * np.pi / 180
    # We chose to use results from Berry's model for this example:
    Intensity_on_pixels = Simu_Micro_Polarizers(Skylight_Relative_Intensity,AoP_Matrix_Global_rad_Berry,DoLP_Matrix_Berry,tolerance_rad,extinction_ratio)

    ## ----- Sensor's simulation -----

    # This part deals with pixels simulation
    # Here we enter the saturation ratio
    # It is the ratio between the sensor irradiance saturation value and the
    # maximum relative irradiance comming on sensor's pixels.
    pixel_saturation = 1.6
    # Here we enter the number of output bits for grayscale
    number_of_bits = 12
    # Here we enter the per pixel gaussian noise Signal to Noise Ratio
    SNR = 125 # Noise Disturbance
    Bits_Matrix = Simu_Sensor(Intensity_on_pixels,pixel_saturation,number_of_bits,SNR)

    ## ----- Data processing -----

    # This part is not a real part of the simulator.
    # It is just about to deal with camera output and compute polarization measurment, just like we would do with a real camera.
    AoP_data_processing_imframe,AoP_data_processing_meridianframe,DoLP_data_processing = Simu_Data_Processing(Bits_Matrix)
    rows_print,cols_print = DoLP_data_processing.shape
    X_mesh_print = np.ones((rows_print,1)) * (np.arange(1,cols_print+1,1))
    Y_mesh_print = ((np.arange(rows_print,1+- 1,- 1)))[:,np.newaxis] * np.ones((1,cols_print))
    # map = cmap('C1')
    # map=plt.colormap('hsv')

    ## ----- Save simulation parameters and polarization outputs -----
    
    # Unique NPZ per iteration
    data_filename = f"polarization_data_iter_{iter:04d}.npz"
    data_filepath = os.path.join(output_dir, data_filename)

    # Save polarization arrays
    np.savez(
        data_filepath,
        AoP_local=AoP_data_processing_meridianframe,
        DoP=DoLP_data_processing,
    )

    # Append CSV row
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            iter,
            time,
            heading,
            Psi_sn,
            zen,
            rotation_axis_azimuth_deg_vec[iter],
            rotation_angle_deg_vec[iter],
            data_filename
        ])
    
    print(f"[INFO] Saved iteration {iter}: {data_filename}")


[INFO] Saved iteration 0: polarization_data_iter_0000.npz
[INFO] Saved iteration 1: polarization_data_iter_0001.npz
[INFO] Saved iteration 2: polarization_data_iter_0002.npz
[INFO] Saved iteration 3: polarization_data_iter_0003.npz
[INFO] Saved iteration 4: polarization_data_iter_0004.npz
[INFO] Saved iteration 5: polarization_data_iter_0005.npz
[INFO] Saved iteration 6: polarization_data_iter_0006.npz
[INFO] Saved iteration 7: polarization_data_iter_0007.npz
[INFO] Saved iteration 8: polarization_data_iter_0008.npz
[INFO] Saved iteration 9: polarization_data_iter_0009.npz
[INFO] Saved iteration 10: polarization_data_iter_0010.npz
[INFO] Saved iteration 11: polarization_data_iter_0011.npz
[INFO] Saved iteration 12: polarization_data_iter_0012.npz
[INFO] Saved iteration 13: polarization_data_iter_0013.npz
[INFO] Saved iteration 14: polarization_data_iter_0014.npz
[INFO] Saved iteration 15: polarization_data_iter_0015.npz
[INFO] Saved iteration 16: polarization_data_iter_0016.npz
[INFO] 